# PyTorch MLP Regression

Train a PyTorch neural network for house-price regression and log the experiment to Weights & Biases.

In [5]:
import copy
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
from src.data_preprocessing import load_house_prices, prepare_features, build_preprocessor
from src.evaluation import regression_metrics, save_metrics
from src.experiment_tracking import start_experiment, log_wandb

In [6]:
DATA_DIR = PROJECT_ROOT / 'data'
EXPERIMENT_ROOT = PROJECT_ROOT / 'experiments'
RANDOM_STATE = 42
torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
train, test = load_house_prices(DATA_DIR)
X, y, X_test = prepare_features(train, test)
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
preprocessor, _, _ = build_preprocessor(X_train)
X_train_encoded = preprocessor.fit_transform(X_train)
X_valid_encoded = preprocessor.transform(X_valid)
X_test_encoded = preprocessor.transform(X_test)

def to_dense_float32(matrix):
    if hasattr(matrix, 'toarray'):
        matrix = matrix.toarray()
    return np.asarray(matrix, dtype=np.float32)

X_train_encoded = to_dense_float32(X_train_encoded)
X_valid_encoded = to_dense_float32(X_valid_encoded)
X_test_encoded = to_dense_float32(X_test_encoded)
target_mean = float(y_train.mean())
target_std = float(y_train.std())
y_train_scaled = ((y_train.to_numpy(dtype=np.float32) - target_mean) / target_std).reshape(-1, 1)
y_valid_scaled = ((y_valid.to_numpy(dtype=np.float32) - target_mean) / target_std).reshape(-1, 1)
print(f'device={device}, input_features={X_train_encoded.shape[1]}')

device=cpu, input_features=272


In [7]:
class HousePriceMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
        )

    def forward(self, features):
        return self.network(features)

train_dataset = TensorDataset(torch.from_numpy(X_train_encoded), torch.from_numpy(y_train_scaled))
valid_features = torch.from_numpy(X_valid_encoded).to(device)
valid_targets = torch.from_numpy(y_valid_scaled).to(device)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
model = HousePriceMLP(X_train_encoded.shape[1]).to(device)
loss_function = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)
best_state = None
best_valid_loss = float('inf')
patience = 30
epochs_without_improvement = 0
history = []

for epoch in range(300):
    model.train()
    train_losses = []
    for features, targets in train_loader:
        features, targets = features.to(device), targets.to(device)
        optimizer.zero_grad()
        loss = loss_function(model(features), targets)
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())

    model.eval()
    with torch.no_grad():
        valid_loss = loss_function(model(valid_features), valid_targets).item()
    history.append({'epoch': epoch + 1, 'train_loss': float(np.mean(train_losses)), 'valid_loss': valid_loss})
    if valid_loss < best_valid_loss - 1e-5:
        best_valid_loss = valid_loss
        best_state = copy.deepcopy(model.state_dict())
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1
    if epochs_without_improvement >= patience:
        break

model.load_state_dict(best_state)
model.eval()
with torch.no_grad():
    valid_scaled_prediction = model(valid_features).cpu().numpy().ravel()
valid_prediction = valid_scaled_prediction * target_std + target_mean
metrics = regression_metrics(y_valid, valid_prediction)
print(f'epochs={len(history)}, best_valid_loss={best_valid_loss:.6f}, metrics={metrics}')

epochs=59, best_valid_loss=0.131874, metrics={'mae': 16385.03515625, 'rmse': 28057.8499532662, 'r2': 0.897365152835846, 'rmsle': 0.13540696495551482}


In [8]:
model_path = None
os.environ['WANDB_MODE'] = 'online'
run_id, run_dir = start_experiment(
    EXPERIMENT_ROOT,
    'pytorch_mlp',
    {
        'model': 'PyTorch MLP',
        'task': 'regression',
        'random_state': RANDOM_STATE,
        'epochs': len(history),
        'input_features': int(X_train_encoded.shape[1]),
    },
)
with torch.no_grad():
    test_features = torch.from_numpy(X_test_encoded).to(device)
    test_scaled_prediction = model(test_features).cpu().numpy().ravel()
test_prediction = test_scaled_prediction * target_std + target_mean
prediction_path = run_dir / 'predictions.csv'
pd.DataFrame({'Id': test['Id'], 'SalePrice': test_prediction}).to_csv(prediction_path, index=False)
metrics_path = run_dir / 'metrics.json'
save_metrics(metrics, metrics_path)
model_path = run_dir / 'model.pt'
torch.save({'model_state_dict': model.state_dict(), 'target_mean': target_mean, 'target_std': target_std}, model_path)
mode = log_wandb(
    run_dir,
    'house-price-regression',
    'pytorch_mlp',
    {'epochs': len(history), 'input_features': int(X_train_encoded.shape[1])},
    metrics,
    [model_path, prediction_path, metrics_path, run_dir / 'config.json'],
)
print(f'Run: {run_dir}')
print(f'W&B mode: {mode}')

mae,▁
r2,▁
rmse,▁
rmsle,▁
mae,16385.03516
r2,0.89737
rmse,28057.84995
rmsle,0.13541


Run: c:\Users\ASUS\Data_Science_ Junior\DL\Tuan02\house_price\experiments\pytorch_mlp\20260919_180218
W&B mode: online
